# EDA on silver layer data to identify exactly what should be cleaned

In [0]:
%python
from pyspark.sql.functions import col, when, trim, regexp_replace, dense_rank, sum as _sum, right, to_date, lit
from pyspark.sql.window import Window

In [0]:
df_bronze = spark.sql("SELECT * FROM marathos.bronze.raw_marathon_data")

In [0]:
df_bronze.printSchema()
df_bronze.display()

## Distance format
- What kind of different event distances are there in this dataset and how do they differ?
    - There are over 2000 ways of writing the DISTANCE of the marathon which is crazy.

In [0]:
df_bronze.groupBy("event_distance_length").count().orderBy(col("count").desc()).display()

## How many event names does NOT end with a country code inside of parenthesis? 
- Example: Ishigaki Island Ultra Marathon (JPN)
    - What the cell proves is that it is alot of cleaning that is needed in this data. Output was:


| event_name | 
| :--- | 
| """6hs Solidarias """"Corré por los Chicos"""" (ARG)""" |
| """The """"James"""" Stampede 50 (NZL)""" |
| """Backyard Ultra """"THE LAST LAP"""" (SUI)""" |
| """24hs Solidarias """"Corré por los Chicos"""" (ARG)""" |
| """Tortugas """"A"""" Mountain 24 Hour Challenge (USA)""" |
| """12hs Solidarias """"Corré por los Chicos"""" (ARG)""" |
| """Ultra Trail Race """"El Chico Hidalgo"""" (MEX)""" |
| """6 Horas individual y posta """"Aniversario Ciudad de Bahía Blanca"""" (ARG)""" |
| """6 ore al Parco Adda-Mallero """"Renato Bartesaghi"""" (ITA)""" |
| """The """"James"""" Stampede 45 (NZL)""" |


- There is alot of garbage quotation signs that needs to be cleaned in this data.

In [0]:
df_bronze.filter(~col("event_name").rlike(r"\([A-Z]{3}\)$")).select("event_name").distinct().limit(20).display()

### Number of invalid rows in relation to d(days) is: 62 727

In [0]:
df_days = df_bronze.filter(col("athlete_performance").rlike("(?i)d"))
print(f"Number of rows that contains days (d): {df_days.count()}")

In [0]:
df_days.select("event_name", "event_distance_length", "athlete_performance").display()

### How many nulls are there in important columns for silver obt?

| nulls_performance | nulls_birth_year | nulls_gender | nulls_speed |
| :--- | :--- | :--- | :--- |
| 2 | 588 161 | 7 | 224 |

- Only 2 rows are missing the athlete performance
- Almost 600k null values for athletes Year of Birth.
- Only 7 nulls on athlete gender
- Only 224 rows with nulls on athletes avarage speed

In [0]:
df_bronze.select(
    _sum(col("athlete_performance").isNull().cast("int")).alias("nulls_performance"),
    _sum(col("athlete_year_of_birth").isNull().cast("int")).alias("nulls_birth_year"),
    _sum(col("athlete_gender").isNull().cast("int")).alias("nulls_gender"),
    _sum(col("athlete_average_speed").isNull().cast("int")).alias("nulls_speed")
).display()

### Find dates that does NOT follow the standard pattern which is DD.MM.YYYY
- over 7k(7340) rows display odd formatting on dates. Example sample is:

| event_dates |
| :--- |
| 24.04.-08.05.2021 |
| 15.09.-02.10.2021 |
| 04.-09.10.2021 |
| 29.-30.01.2022 |
| 13.-14.08.2022 |
| 28.-29.10.2022 |
| 10.-20.03.1997 |
| 16.-17.10.2021 |
| 30.09.-02.10.2021 |
| 29.-30.12.2021 |
| 31.05.-03.06.2022 |


In [0]:
print("Deviations in event_dates:")
(df_bronze.filter(~col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$")).select("event_dates").distinct().display())

### Attempt to salvage all 7356 event_date rows and convert them to DateTypes
- with sparks import of `lit` i should be able to use sparks `right()` function I should be able to clean and convert all odd dates to DateTypes.
    - By importing `lit` and wrapping my 10 integer with it and using `right()` i could convert it to a literal column and make sense of my messy `event_dates` rows.


- The results of my attempt is:


| event_dates | clean_date_string | official_event_date | 
| :--- | :--- | :--- |
| 29.-30.05.2021 | 30.05.2021 | 2021-05-30 |
| 16.-18.04.2021 | 18.04.2021 | 2021-04-18 | 
| 30.04.-01.05.2021 | 01.05.2021 | 2021-05-01 |

    

In [0]:
# Pick out the last 10 chars from its string that should give me an end date
df_dates_test = df_bronze.withColumn("clean_date_string", right(col("event_dates"), lit(10)))

# Converting that string to a real date YYYY-MM-DD
df_dates_test = df_dates_test.withColumn("official_event_date", to_date(col("clean_date_string"), "dd.MM.yyyy"))


# Show the result (filtering on strings that contains - to see anomalies in data)
print("Conversion of messy dates:")
(df_dates_test
 .filter(col("event_dates").contains("-"))
 .select("event_dates", "clean_date_string", "official_event_date")
 .display())

### Finding anomalies in event_distance_length and athlete_performance columns
- Trying to find outliers in these columns that arent `km`, `mi` or `h` that my pipeline will miss if I dont know about them.

